# Reaktive Maschinenagenten mit Mesa (v3.x)

In diesem Notebook wirst du:
- **Mesa 3.x** mit Visualisierungsunterstützung installieren,
- einen **reaktiven Agenten** implementieren, der die Temperatur einer Maschine überwacht,
- ein einfaches **Fabrikmodell** mit mehreren Maschinenagenten aufbauen,
- eine **interaktive Visualisierung** direkt im Notebook starten und verschiedene Parameter erkunden,
- das Modell um **Wartungs-** und **Auftragsagenten** erweitern.


## 1. Installation
Zunächst installieren wir Mesa **3.0.3** inklusive Visualisierungsunterstützung.


In [ ]:
#!pip install -U mesa[viz]==3.0.3 altair==5.2.0 networkx solara

## 2. Mesa-Grundlagen
Mesa-Modelle bestehen aus drei Hauptteilen:
- einer **Model**-Klasse, die den globalen Zustand verwaltet,
- einer oder mehreren **Agent**-Klassen, die das Verhalten einzelner Agenten definieren,
- einer optionalen **Visualisierung**, um das Modell während der Ausführung zu beobachten.

**Wichtige Änderungen in Mesa 3.x gegenüber v2.x:**
- `Agent.__init__` braucht keine `unique_id` mehr – sie wird automatisch vergeben.
- Den Scheduler (`RandomActivation`) gibt es nicht mehr → stattdessen `self.agents.shuffle_do("step")`.
- `model.schedule.agents` → `model.agents`.
- Agenten entfernen: `self.remove()` statt `self.model.schedule.remove(self)`.


## 3. Implementierung des `MachineAgent`
Der `MachineAgent` repräsentiert eine einzelne Maschine in einer Fabrik.


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Reaktiver Agent, der eine Maschine repräsentiert und deren Temperatur überwacht.

    Regel:
        temperature > threshold            -> Zustand = 'HOT'
        temperature > 0.75 * threshold     -> Zustand = 'WARM'
        temperature < 20                   -> Zustand = 'COOL'
        sonst                              -> Zustand = 'OK'
    """

    def __init__(self, model, threshold=70):
        super().__init__(model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False

    def sense_temperature(self):
        """Einfaches Sensormodell: Temperatur + zufälliges Rauschen."""
        if self.state != "HOT":
            noise = random.uniform(-1, 2)
            self.temperature += noise

    def decide(self):
        """Reaktive Entscheidungsregel, die nur auf der aktuellen Temperatur basiert."""
        if self.temperature > self.threshold:
            self.state = "HOT"
        elif self.temperature > 0.75 * self.threshold:
            self.state = "WARM"
        elif self.temperature < 20:
            self.state = "COOL"
        else:
            self.state = "OK"

    def act(self):
        """Maschine abkühlen, wenn sie überhitzt ist."""
        if self.state == "HOT":
            self.temperature -= 10

    def step(self):
        """Agentenschritt = wahrnehmen → entscheiden → handeln."""
        self.sense_temperature()
        self.decide()
        self.act()


## 4. Implementierung des `FactoryModel`
Das Modell platziert mehrere Maschinen auf einem Gitter und aktiviert sie in
zufälliger Reihenfolge. Ein **DataCollector** verfolgt, wie viele Maschinen sich
in den Zuständen `"HOT"` und `"WARM"` befinden.


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "HOT")


def count_warm_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "WARM")


class FactoryModel(Model):
    """Einfaches Fabrikmodell mit einem Gitter aus Maschinenagenten."""

    def __init__(self, width=10, height=10, density=0.3, threshold=70, seed=None):
        super().__init__(seed=seed)
        self.width = width
        self.height = height
        self.density = density
        self.threshold = threshold

        self.grid = MultiGrid(width, height, torus=False)

        for x in range(self.width):
            for y in range(self.height):
                if self.random.random() < self.density:
                    agent = MachineAgent(self, threshold=self.threshold)
                    self.grid.place_agent(agent, (x, y))

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines":  count_hot_machines,
                "WarmMachines": count_warm_machines,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


## 5. Visualisierung mit Mesa 3.x (SolaraViz)
Mesa 3.x verwendet **SolaraViz** als Visualisierungsframework.

Die `post_process`-Funktion wird nach dem Zeichnen der Agenten aufgerufen und
erlaubt beliebige Matplotlib-Anpassungen. Wichtig: Sie muss als **Closure**
definiert werden, damit sie immer auf das *aktuelle* Modellobjekt zugreift –
nicht auf eine fest verdrahtete globale Variable.

⚠️ **Hinweis:** Falls die Anzeige im Notebook nicht direkt funktioniert,
speichere den Code in eine `.py`-Datei und starte sie mit `solara run datei.py`.


In [ ]:
from mesa.visualization import SolaraViz, make_plot_component, make_space_component


# ── Agentendarstellung ─────────────────────────────────────────────────────────
def agent_portrayal(agent):
    return {
        "color": {
            "HOT":  "red",
            "WARM": "orange",
            "COOL": "skyblue",
            "OK":   "green",
        }.get(agent.state, "gray"),
        "size":   600,
        "marker": "s",
        "alpha":  0.9,
    }


# ── Closure: post_process erhält das Modell über den Parameter ─────────────────
# SolaraViz erstellt bei Parameteränderungen ein NEUES Modellobjekt.
# Damit post_process immer das aktuelle Modell sieht, übergeben wir es
# über eine Closure – NICHT als globale Variable.
def make_post_process(model):
    def post_process(ax):
        ax.figure.set_size_inches(6, 6)

        legend = ax.get_legend()
        if legend:
            legend.remove()

        for agent in model.agents:
            if not isinstance(agent, MachineAgent):
                continue
            x, y = agent.pos
            ax.text(
                x, y,
                f"{agent.temperature:.1f}°C",
                ha="center", va="center",
                fontsize=7, color="white", fontweight="bold",
            )

        ax.grid(True, alpha=0.3)
    return post_process


# ── Komponenten ────────────────────────────────────────────────────────────────
def make_components(model):
    """Erzeugt space_component und plot_component für das gegebene Modell."""
    space = make_space_component(
        agent_portrayal,
        post_process=make_post_process(model),
    )
    plot = make_plot_component(
        {"HotMachines": "red", "WarmMachines": "orange"}
    )
    return space, plot


# ── Modellparameter ────────────────────────────────────────────────────────────
model_params = {
    "width":  10,
    "height": 10,
    "density": {
        "type":  "SliderFloat",
        "value": 0.3,
        "label": "Dichte",
        "min":   0.1,
        "max":   1.0,
        "step":  0.1,
    },
    "threshold": {
        "type":  "SliderInt",
        "value": 70,
        "label": "Temperatur-Schwelle",
        "min":   30,
        "max":   100,
        "step":  5,
    },
}

# ── Modellinstanz und Visualisierung ───────────────────────────────────────────
model_instance = FactoryModel(width=10, height=10, density=0.3, threshold=70)
space_component, plot_component = make_components(model_instance)

page = SolaraViz(
    model_instance,
    components=[space_component, plot_component],
    model_params=model_params,
    name="Reaktive Maschinenagenten",
)
page


## 6. Simulation ohne Visualisierung testen
Falls du nur die Modelllogik überprüfen möchtest, kannst du das Modell auch direkt einige Schritte laufen lassen.


In [ ]:
model = FactoryModel(width=10, height=10, density=0.3, threshold=70, seed=42)
for _ in range(20):
    model.step()

model.datacollector.get_model_vars_dataframe().tail()


## 7. Erkundungsaufgaben (für das Labor)
Nutze das laufende Modell, um das Verhalten der reaktiven Agenten zu erkunden:

1. **Schwellenwert anpassen**
   - Starte mit `threshold=70` und probiere dann 60 oder 80.
   - Was passiert mit der Anzahl der `"HOT"`-Maschinen im Zeitverlauf?

2. **Dichte verändern**
   - Erhöhe oder verringere den `density`-Parameter.
   - Wie beeinflusst das den Gesamtzustand des Systems?

3. **Zustandslogik erweitern (optional)**
   - Füge weitere Zustände hinzu, z. B. `"KRITISCH"`, und mappe sie auf neue Farben in `agent_portrayal`.
   - Definiere eigene Temperaturbereiche für jeden Zustand.

4. **(Fortgeschritten) Wartungsagenten hinzufügen**
   - Erstelle einen zweiten Agententyp, der durch das Gitter wandert und überhitzte
     Maschinen „repariert", indem er deren Temperatur und Zustand zurücksetzt.

5. **(Fortgeschritten) Auftragsagenten hinzufügen**
   - Erstelle einen dritten Agententyp, der Aufträge durch das Gitter bewegt und
     selbstständig freie Maschinen sucht.


## 8. Musterlösungen für die Fortgeschrittenen-Aufgaben
Wir erweitern das Modell um zwei weitere Agententypen:

1. **`MaintenanceAgent`** – wandert durch das Gitter und repariert überhitzte Maschinen.
2. **`OrderAgent`** – repräsentiert einen Produktionsauftrag, sucht eine freie Maschine
   und entfernt sich mit `self.remove()` selbst aus dem Modell (Mesa 3.x).


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Erweiterter reaktiver Maschinenagent (identisch mit oben)."""

    def __init__(self, model, threshold=70):
        super().__init__(model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False

    def sense_temperature(self):
        if self.state != "HOT":
            self.temperature += random.uniform(-1, 2)

    def decide(self):
        if self.temperature > self.threshold:
            self.state = "HOT"
        elif self.temperature > 0.75 * self.threshold:
            self.state = "WARM"
        elif self.temperature < 20:
            self.state = "COOL"
        else:
            self.state = "OK"

    def act(self):
        pass  # Abkühlung erfolgt durch den MaintenanceAgent

    def step(self):
        self.sense_temperature()
        self.decide()
        self.act()


class MaintenanceAgent(Agent):
    """Wandert durch das Gitter und repariert HOT-Maschinen."""

    def step(self):
        neighbours = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=True
        )
        self.model.grid.move_agent(self, random.choice(neighbours))

        for agent in self.model.grid.get_cell_list_contents([self.pos]):
            if isinstance(agent, MachineAgent) and agent.state == "HOT":
                agent.temperature = agent.threshold - 40
                agent.state = "OK"
                agent.busy = False

class OrderAgent(Agent):
    """Produktionsauftrag: sucht eine freie Maschine, belegt sie für 'duration' Schritte."""

    def __init__(self, model, duration=10):
        super().__init__(model)
        self.duration = duration
        self.assigned_machine = None   # ← Referenz auf die belegte Maschine

    def step(self):
        # ── Fall 1: noch keine Maschine gefunden → suchen ─────────────────────
        if self.assigned_machine is None:
            neighbours = self.model.grid.get_neighborhood(
                self.pos, moore=True, include_center=True
            )
            self.model.grid.move_agent(self, random.choice(neighbours))

            for agent in self.model.grid.get_cell_list_contents([self.pos]):
                if isinstance(agent, MachineAgent) and agent.state != "HOT" and not agent.busy:
                    agent.busy = True
                    agent.temperature += 5
                    self.assigned_machine = agent   # ← Referenz merken
                    break

        # ── Fall 2: Maschine belegt → Dauer herunterzählen ────────────────────
        else:
            self.duration -= 1
            for agent in self.model.grid.get_cell_list_contents([self.pos]):
                if isinstance(agent, MachineAgent) and agent.state != "HOT":
                    agent.temperature += 5 
            if self.duration <= 0:
                self.assigned_machine.busy = False
                self.model.grid.remove_agent(self)  # ← erst aus dem Grid
                self.remove()


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "HOT")

def count_warm_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "WARM")

def count_orders(model):
    return sum(1 for a in model.agents if isinstance(a, OrderAgent))


class FactoryModelExtended(Model):
    """Erweitertes Fabrikmodell mit Maschinen-, Wartungs- und Auftragsagenten."""

    def __init__(self, width=10, height=10, density=0.3, threshold=70,
                 n_maintenance=2, n_orders=5, seed=None):
        super().__init__(seed=seed)
        self.width = width
        self.height = height
        self.grid = MultiGrid(width, height, torus=False)

        for _ in range(n_orders):
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(OrderAgent(self), pos)

        for x in range(width):
            for y in range(height):
                if self.random.random() < density:
                    self.grid.place_agent(MachineAgent(self, threshold=threshold), (x, y))

        for _ in range(n_maintenance):
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(MaintenanceAgent(self), pos)

        

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines":  count_hot_machines,
                "WarmMachines": count_warm_machines,
                "Auftraege":    count_orders,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


### Visualisierung des erweiterten Modells
Verschiedene Agententypen werden mit unterschiedlichen Formen und Farben dargestellt:
- **Maschinen** – farbige Quadrate mit Temperaturanzeige
- **Wartungsagenten** – türkise Kreise mit „W"
- **Auftragsagenten** – schwarze kleine Quadrate mit Auftrags-ID


In [ ]:
from mesa.visualization import SolaraViz, make_plot_component, make_space_component

# Modellparameter festlegen
DENSITY = 0.2 
THRESHOLD = 70 
N_MAINTENANCE = 2 
N_ORDERS = 30


# ── Agentendarstellung ─────────────────────────────────────────────────────────
def agent_portrayal_ext(agent):
    if isinstance(agent, MachineAgent):
        return {
            "color": {
                "HOT":  "red",
                "WARM": "orange",
                "COOL": "skyblue",
                "OK":   "green",
            }.get(agent.state, "gray"),
            "size": 600, "marker": "s", "alpha": 0.7,
        }
    if isinstance(agent, MaintenanceAgent):
        return {"color": "cyan",  "size": 300, "marker": "o", "alpha": 0.95}
    if isinstance(agent, OrderAgent): 
        return {"color": "black", "size": 200, "marker": "s", "alpha": 0.85}
    return {}


# ── Closure für post_process ───────────────────────────────────────────────────
def make_post_process_ext(model):
    def post_process(ax):
        ax.figure.set_size_inches(6, 6)
        legend = ax.get_legend()
        if legend:
            legend.remove()

        for agent in model.agents:
            x, y = agent.pos
            if isinstance(agent, MachineAgent):
                ax.text(x, y, f"{agent.temperature:.1f}°C",
                        ha="center", va="center",
                        fontsize=6, color="white", fontweight="bold")
            elif isinstance(agent, MaintenanceAgent):
                ax.text(x, y, "W",
                        ha="center", va="center",
                        fontsize=9, color="black", fontweight="bold")
            elif isinstance(agent, OrderAgent):
                ax.text(x, y, str(agent.unique_id),
                        ha="center", va="center",
                        fontsize=6, color="white")

        ax.grid(True, alpha=0.3)
    return post_process


''' ── Modellparameter ────────────────────────────────────────────────────────────
model_params_ext = {
    "width": 10, "height": 10,
    "density": {
        "type": "SliderFloat", "value": DENSITY,
        "label": "Maschinendichte", "min": 0.1, "max": 1.0, "step": 0.1,
    },
    "threshold": {
        "type": "SliderInt", "value": THRESHOLD,
        "label": "Temperatur-Schwelle", "min": 30, "max": 100, "step": 5,
    },
    "n_maintenance": {
        "type": "SliderInt", "value": N_MAINTENANCE,
        "label": "Wartungsagenten", "min": 0, "max": 10, "step": 1,
    },
    "n_orders": {
        "type": "SliderInt", "value": N_ORDERS,
        "label": "Auftragsagenten", "min": 10, "max": 100, "step": 1,
    },
}
'''

# ── Modellinstanz und Visualisierung ───────────────────────────────────────────
model_ext = FactoryModelExtended(
    width=10, height=10, density=DENSITY, threshold=THRESHOLD, n_maintenance=N_MAINTENANCE, n_orders=N_ORDERS
)

space_ext = make_space_component(
    agent_portrayal_ext,
    post_process=make_post_process_ext(model_ext),
)
plot_ext = make_plot_component(
    {"HotMachines": "red", "WarmMachines": "orange", "Auftraege": "black"}
)

page_ext = SolaraViz(
    model_ext,
    components=[space_ext, plot_ext],
   # model_params=model_params_ext,
    name="Erweitertes Fabrikmodell",
    play_interval=200,   # ← 200ms zwischen Steps = 5 fps
)
page_ext
